# grayscale correlation
projecting down the results fromm 3 to 1 dimension using PCA. Then correlating the result with the individual input variables.

1. load model
2. load dataset
3. predict results using NOFE and other methods
4. project to grayscale
5. evaluate correlation

---
### Utilities

In [1]:
import sys
import json
import random
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import napari
from scipy.interpolate import griddata
from pathlib import Path

from plot_utils_era5 import *

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

sys.path.append(f"{project_root}src/")
# Import utilities
from utilities import KernelNN, KernelNN_subgraph, GraphLMDBReader, setup_device, get_output_affinities
from plot_utilities import plot_results_napari, plot_results_matplotlib

sys.path.append(f"{project_root}/ERA5/scripts/")
from utilities_era5 import ERA5_Graph_Generator as Data_Generator
from utilities_era5 import ERA5_Graph_Generator_perSampleNorm as Data_Generator_old
data_path = project_root / 'ERA5' / 'data'

# Device setup
device = setup_device()
print(f'Using device: {device}')

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import subprocess

def load_model(id:str, model_type:str='random_crop'):
    model_path = project_root / 'ERA5' / 'trained_models' / model_type / f'{id}.pth'
    config_path = project_root / 'ERA5' / 'trained_models' / model_type / 'logs' / 'configs' / f'{id}.json' 
    
    with open(config_path, 'r') as f:
        config = json.load(f)
    
    model = KernelNN(
        width=config['width'],
        ker_width=config['ker_width'],
        depth=config['depth'],
        ker_in=config['edge_attr'],
        in_width=config['node_features'],
        out_width=config['out_width']
    ).to(device)
    
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    return model, config

using Device: mps
Using device: mps


----
### load model + dataset 

In [2]:
model, config = load_model(id='reg_median_woPos_model1_2026-04-17_t11-07-17' , model_type='regular')

In [3]:
lat_min = config["lat_min"]
lat_max = config["lat_max"]
lon_min = config["lon_min"]
lon_max = config["lon_max"]
#N = config["N"]
k = config["k"]
perplexity = config["perplexity"]
sampling_method = config["sampling_method"]
train_date_start_end = config["train_date_start_end"]
test_date_start_end = config["test_date_start_end"]
train_samples_per_date = config["train_samples_per_date"]
test_samples_per_date = config["test_samples_per_date"]
feature_mode = config["feature_mode"]
vars = config["vars"]
sigma_method = config["sigma_method"]

In [4]:
date = "2019-01-10"

bbox = lon_min, lat_min, lon_max, lat_max

#Generator = Data_Generator_old(data_path)
Generator = Data_Generator(data_path)
Generator.load_data(bbox=bbox, vars=vars, filename=f'{date}.nc')
print(f'total points available: {Generator.da.shape[1]*Generator.da.shape[2]}')

N=5000
print('generate data')
graph = Generator.sample_graph(N=N, k=k, perplexity=perplexity, sampling_method=sampling_method, gen_feature_mode=feature_mode, sigma_method=sigma_method)
sample = graph.to(device)

locations = pd.DataFrame({"lat": sample.meta.gps[:,0], "lon": sample.meta.gps[:,1]})
X = sample.normed_values.detach().cpu().numpy()

total points available: 32761
generate data


---
## Get Predictions

In [6]:
# NOFE
print('NOFE ...', end='\r')
pred_nofe = model(sample).detach().cpu().numpy()
print('NOFE\t- done')

# PCA
print('PCA ...', end='\r')
pca = PCA(n_components=3)
pred_pca = pca.fit_transform(X)
print('PCA\t- done')

# t-SNE
print('t-SNE ...', end='\r')
pred_tsne = TSNE(
    n_components=3,
    perplexity=30,
    learning_rate="auto",
    random_state=42
).fit_transform(X)
print('t-SNE\t- done')

# UMAP
print('UMAP ...', end='\r')
umap_id = 'umap_buffer'
dir = f'{project_root}/ERA5/umap'
np.save(f'{dir}/{umap_id}.npy', X)
subprocess.run(f"cd {dir} && conda run -n umap python execute_umap.py {umap_id}", shell=True)
pred_umap = np.load(f'{dir}/res_{umap_id}.npy')

os.remove(f'{dir}/{umap_id}.npy')
os.remove(f'{dir}/res_{umap_id}.npy')
print('UMAP\t- done')

NOFE - done
PCA - done
t-SNE - done
importing libraries
reading in data
applying umap
saving file
"
file saved as res_umap_buffer.npy"

UMAP - done


/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")



---
## Grayscale Projection

In [7]:
pca = PCA(n_components=1)
grayscale_nofe = pca.fit_transform(pred_nofe).squeeze() 
grayscale_pca = pca.fit_transform(pred_pca).squeeze() 
grayscale_tsne = pca.fit_transform(pred_tsne).squeeze() 
grayscale_umap = pca.fit_transform(pred_umap).squeeze() 

---
## Correlate

In [8]:
correlations_nofe = np.array([
    np.abs(np.corrcoef(grayscale_nofe, X[:, i])[0, 1])
    for i in range(X.shape[1])
])

correlations_pca = np.array([
    np.abs(np.corrcoef(grayscale_pca, X[:, i])[0, 1])
    for i in range(X.shape[1])
])

correlations_tsne = np.array([
    np.abs(np.corrcoef(grayscale_tsne, X[:, i])[0, 1])
    for i in range(X.shape[1])
])

correlations_umap = np.array([
    np.abs(np.corrcoef(grayscale_umap, X[:, i])[0, 1])
    for i in range(X.shape[1])
])

In [9]:
corrs = pd.DataFrame({
    'nofe' : correlations_nofe,
    'pca' : correlations_pca,
    'tsne' : correlations_tsne,
    'umap' : correlations_umap
}, index=config['vars'])
print(corrs)
print()
print('Mean Values:')
print(f'NOFE:\t{corrs.nofe.mean():.2f}')
print(f'PCA:\t{corrs.pca.mean():.2f}')
print(f't-SNE:\t{corrs.tsne.mean():.2f}')
print(f'UMAP:\t{corrs.umap.mean():.2f}')

        nofe       pca      tsne      umap
w   0.374892  0.174499  0.003996  0.144082
v   0.051600  0.179870  0.024619  0.002259
z   0.097230  0.579440  0.739427  0.475433
q   0.718328  0.740452  0.306346  0.504174
r   0.740366  0.464637  0.288239  0.113952
t   0.215200  0.624777  0.883055  0.910253
d   0.147504  0.070782  0.006201  0.011006
o3  0.496421  0.743692  0.092029  0.059256
pv  0.417936  0.708714  0.154053  0.042602
u   0.057254  0.062584  0.224082  0.361552
vo  0.541102  0.831661  0.185254  0.266237

Mean Values:
NOFE:	0.35
PCA:	0.47
t-SNE:	0.26
UMAP:	0.26


---
# Run on full scale datset

In [5]:
config['dataset_id']

'toyset_reg_fixcrop_median_woPos'

In [6]:
dataset = GraphLMDBReader(f'{project_root}/ERA5/datasets/regular/{config["dataset_id"]}_test.lmdb')
print(f'loaded {len(dataset)} samples')

loaded 100 samples


In [8]:
corr_res_nofe = []

#for s in range(len(reader)):
for s in range(10):
    print(f'{s+1}/{len(dataset)}', end='\r')

    sample = dataset[s].to(device='mps')
    X = sample.normed_values.detach().cpu().numpy()
    # =================== 
    # NOFE
    # =================== 
    pred_nofe = model(sample).detach().cpu().numpy()
    pca = PCA(n_components=1)
    grayscale_nofe = pca.fit_transform(pred_nofe).squeeze() 
    correlations_nofe = np.array([
        np.abs(np.corrcoef(grayscale_nofe, X[:, i])[0, 1])
        for i in range(X.shape[1])
    ])
    corr_res_nofe.append(correlations_nofe)

corr_res_nofe = np.vstack(corr_res_nofe)
print(corr_res_nofe.mean())

0.40091236276760883


In [14]:
corr_res_pca = []

#for s in range(len(reader)):
for s in range(10):
    print(f'{s+1}/{len(dataset)}', end='\r')

    sample = dataset[s].to(device='mps')
    X = sample.normed_values.detach().cpu().numpy()
    # =================== 
    # PCA
    # =================== 
    pca = PCA(n_components=3)
    pred_pca = pca.fit_transform(X)
    pca = PCA(n_components=1)
    grayscale_pca = pca.fit_transform(pred_pca).squeeze() 
    correlations_pca = np.array([
        np.abs(np.corrcoef(grayscale_pca, X[:, i])[0, 1])
        for i in range(X.shape[1])
    ])
    corr_res_pca.append(correlations_pca)

corr_res_pca = np.vstack(corr_res_pca)    

In [15]:
corr_res_tsne = []

#for s in range(len(reader)):
for s in range(10):
    print(f'{s+1}/{len(dataset)}', end='\r')

    sample = dataset[s].to(device='mps')
    X = sample.normed_values.detach().cpu().numpy()
    # =================== 
    # t-SNE
    # =================== 
    pred_tsne = TSNE(
        n_components=3,
        perplexity=30,
        learning_rate="auto",
        random_state=42
    ).fit_transform(X)
    pca = PCA(n_components=1)
    grayscale_tsne = pca.fit_transform(pred_tsne).squeeze() 
    correlations_tsne = np.array([
        np.abs(np.corrcoef(grayscale_tsne, X[:, i])[0, 1])
        for i in range(X.shape[1])
    ])
    corr_res_tsne.append(correlations_tsne)

corr_res_tsne = np.vstack(corr_res_tsne)

In [16]:
corr_res_umap = []

#for s in range(len(reader)):
for s in range(10):
    print(f'{s+1}/{len(dataset)}', end='\r')

    sample = dataset[s].to(device='mps')
    X = sample.normed_values.detach().cpu().numpy()
    # =================== 
    # UMAP
    # =================== 
    umap_id = 'umap_buffer'
    dir = f'{project_root}/ERA5/umap'
    np.save(f'{dir}/{umap_id}.npy', X)
    subprocess.run(f"cd {dir} && conda run -n umap python execute_umap.py {umap_id}", shell=True)
    pred_umap = np.load(f'{dir}/res_{umap_id}.npy')
    os.remove(f'{dir}/{umap_id}.npy')
    os.remove(f'{dir}/res_{umap_id}.npy')
    pca = PCA(n_components=1)
    grayscale_umap = pca.fit_transform(pred_umap).squeeze() 
    correlations_umap = np.array([
        np.abs(np.corrcoef(grayscale_umap, X[:, i])[0, 1])
        for i in range(X.shape[1])
    ])
    corr_res_umap.append(correlations_umap)

corr_res_umap = np.vstack(corr_res_umap)

importing libraries
reading in data
applying umap
saving file
"
file saved as res_umap_buffer.npy"



/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")



importing libraries
reading in data
applying umap
saving file
"
file saved as res_umap_buffer.npy"



/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")



importing libraries
reading in data
applying umap
saving file
"
file saved as res_umap_buffer.npy"



/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")



... (10 outputs left out)

importing libraries
reading in data
applying umap
saving file
"
file saved as res_umap_buffer.npy"



/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")



importing libraries
reading in data
applying umap
saving file
"
file saved as res_umap_buffer.npy"



/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")



In [17]:
corrs = pd.DataFrame({
    'nofe' : corr_res_nofe.mean(axis=0),
    'pca' : corr_res_pca.mean(axis=0),
    'tsne' : corr_res_tsne.mean(axis=0),
    'umap' : corr_res_umap.mean(axis=0)
}, index=config['vars'])

print('Mean correlation per variable:')
print(corrs)

print('\nMean correlation coefficient:\n')
print(f'NOFE:\t{corr_res_nofe.mean():.2f}')
print(f'PCA:\t{corr_res_pca.mean():.2f}')
print(f't-SNE:\t{corr_res_tsne.mean():.2f}')
print(f'UMAP:\t{corr_res_umap.mean():.2f}')

Mean correlation per variable:
        nofe       pca      tsne      umap
w   0.448350  0.107363  0.049260  0.018821
v   0.148338  0.161247  0.061178  0.145916
z   0.068783  0.869065  0.859647  0.907555
q   0.652030  0.658879  0.467522  0.420357
r   0.751602  0.082217  0.153559  0.260441
t   0.046294  0.918947  0.903673  0.949008
d   0.039285  0.026751  0.018860  0.010368
o3  0.206824  0.142630  0.333545  0.342171
pv  0.109957  0.398305  0.188463  0.179591
u   0.163668  0.535948  0.450053  0.415513
vo  0.106900  0.448104  0.107213  0.169618

Mean correlation coefficient:

NOFE:	0.25
PCA:	0.40
t-SNE:	0.33
UMAP:	0.35
